# 🧠 SwiGLU: What are w_gate, w_up, and w_down?

Welcome! In this notebook, we break down the 3 matrices of **SwiGLU** with a clear diagram and an intuitive real-world example.

---

## 1. Where Does the Word Get Its Context?

A single word like <b>"insert"</b> cannot know its meaning alone.<br>
In a Transformer layer, <b>Self-Attention runs BEFORE SwiGLU</b>:

<pre style='font-size: 14px; background: #0d1117; color: #58a6ff; padding: 16px; border-radius: 8px; border: 1px solid #30363d;'>
Input Tokens: ["class", "BinaryTree:", "def", "insert"]
                      │
                      ▼
            ┌───────────────────┐
            │   Self-Attention  │  <-- STEP 1: CONTEXT GATHERING
            └─────────┬─────────┘      Looks back at "BinaryTree"
                      │
                      ▼
   Vector x NOW CONTAINS CONTEXT!
   ("insert" now carries the flavor of "BinaryTree")
                      │
                      ▼
            ┌───────────────────┐
            │      SwiGLU       │  <-- STEP 2: FACT RECALL & FILTERING
            └───────────────────┘
</pre>

---
## 2. The 3 Matrices Diagram

<pre style='font-size: 14px; background: #0d1117; color: #c9d1d9; padding: 16px; border-radius: 8px; border: 1px solid #30363d;'>
                           ┌─────────────────────────┐
                           │   Contextual Vector x   │  (Size: 768)
                           │   ("insert in tree")    │
                           └───────────┬─────────────┘
                                       │
                      ┌────────────────┴────────────────┐
                      ▼                                 ▼
             [ W_up Matrix ]                   [ W_gate Matrix ]
             "The Consultant"                 "The Fact-Checker"
                      │                                 │
                      ▼                                 ▼
         Generates Candidates:               Calculates Relevance:
         [ DB query, List, Tree ]            [ -3.0,   -2.0,   4.0 ]
                      │                                 │
                      │                                 ▼
                      │                         [ SiLU Activation ]
                      │                         Turns into volume dials:
                      │                         [ 0%,    0%,   100% ]
                      │                                 │
                      └────────────────┬────────────────┘
                                       ▼
                              [ ⊙ Multiplier ]
                           Multiplies candidates by dials:
                           DB query x   0% = 0.0 (Silenced!)
                           List     x   0% = 0.0 (Silenced!)
                           Tree     x 100% = Tree Concept (Kept!)
                                       │
                                       ▼
                               [ W_down Matrix ]
                               "The Summarizer"
                               (Compresses 2048 back to 768)
                                       │
                                       ▼
                           ┌─────────────────────────┐
                           │   Clean Output Vector   │  (Size: 768)
                           └─────────────────────────┘
</pre>

---
## 3. How Does Training "Force" These Roles?

We do not write any code telling <b>w_gate</b> to be a filter or <b>w_up</b> to be a candidate generator.<br>
Their roles emerge naturally from their mathematical structure:

<ol>
  <li><b>Path A has SiLU:</b> The SiLU curve squashes negative numbers down toward zero. It is mathematically optimized to act as a multiplier (volume knob).</li>
  <li><b>Path B is Linear:</b> It has no squashing function, so it can output large, expressive positive and negative feature values.</li>
  <li><b>The Gradient Push:</b> If irrelevant noise causes the model to guess the wrong word, backpropagation automatically pushes the weights of <b>w_gate</b> so its output drops to <b>0.0</b>, silencing that neuron to minimize loss!</li>
</ol>

---
## 4. Hands-On Pencil & Paper: See the Filtering in Action 📝

Suppose our candidate projection produces 3 features:
<pre style='font-size: 15px; background: #161b22; color: #7ee787; padding: 12px; border-radius: 8px;'>
w_up(x) = [ 10.0,   20.0,   -5.0 ]
           (DB query) (List)   (Tree)
</pre>

And our gate decides that only the Tree feature is relevant right now:
<pre style='font-size: 15px; background: #161b22; color: #bc8cff; padding: 12px; border-radius: 8px;'>
SiLU(w_gate(x)) = [ 0.0,   0.0,   1.0 ]
                   (Off)   (Off)  (On!)
</pre>

### Calculate the Element-Wise Product (⊙) on Paper:
1. Coordinate 1: 10.0 · 0.0 = <b>0.0</b> &nbsp; (Silenced!)
2. Coordinate 2: 20.0 · 0.0 = <b>0.0</b> &nbsp; (Silenced!)
3. Coordinate 3: -5.0 · 1.0 = <b>-5.0</b> &nbsp; (Preserved!)

<br>
<b>Filtered Knowledge Vector:</b> [ 0.0,  0.0,  -5.0 ]

In [ ]:
import torch

candidates = torch.tensor([10.0, 20.0, -5.0])
gate_dials = torch.tensor([0.0, 0.0, 1.0])

filtered_vector = gate_dials * candidates
print(f"Candidate features:        {candidates.tolist()}")
print(f"Gate volume dials:         {gate_dials.tolist()}")
print(f"Filtered Knowledge Vector: {filtered_vector.tolist()}")

---
## 5. Run Our Actual PyTorch SwiGLU Layer

Now let's trace this through our project's actual `SwiGLUFFN` class from `src/model/mlp.py`:

In [ ]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.model.mlp import SwiGLUFFN

dim = 768
ffn = SwiGLUFFN(dim=dim)

print(f"w_up matrix shape:   {list(ffn.w_up.weight.shape)}   (projects 768 -> 2048)")
print(f"w_gate matrix shape: {list(ffn.w_gate.weight.shape)} (projects 768 -> 2048)")
print(f"w_down matrix shape: {list(ffn.w_down.weight.shape)} (projects 2048 -> 768)")

# Pass a token vector through the full forward pass
x = torch.randn(1, 1, dim)
output = ffn(x)
print(f"\nInput Shape:  {list(x.shape)}")
print(f"Output Shape: {list(output.shape)}")
assert output.shape == x.shape
print("SUCCESS: SwiGLU forward pass fully traced!")